# Academy Object Detection

Runs on Google Colab and Kaggle Notebooks. Enable a GPU first — Colab: **Runtime > Change
runtime type**; Kaggle: **Settings > Accelerator**. On Kaggle also switch **Settings >
Internet** on, otherwise the clone, the installs, and the dataset download all fail.

This notebook only prepares the environment and calls the repository scripts; the training and
evaluation code stays in the Python files.

## Clone the repository

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle").exists()

WORK_ROOT = Path("/content") if IN_COLAB else Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
REPO_DIR = WORK_ROOT / "AcademyObjectDetection"

if not REPO_DIR.exists():
    !git clone https://github.com/malek-wahidi/AcademyObjectDetection.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f"working directory: {Path.cwd()}")

## Install the extra packages

Colab and Kaggle already provide PyTorch and Torchvision built for their drivers, so only the
project's other dependencies are installed here.

In [ ]:
%pip install -q einops "torchmetrics[detection]" pyyaml wandb

## Choose where data and runs are stored

On Colab the dataset and checkpoints go to Google Drive so they survive a disconnect. On Kaggle
they go to the session's working directory — save it as a Kaggle Dataset if you want to keep it,
and note the working directory has a size limit.

In [ ]:
USE_WANDB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    STORAGE = Path("/content/drive/MyDrive/AcademyObjectDetection")
else:
    STORAGE = WORK_ROOT / "academy-object-detection"

STORAGE.mkdir(parents=True, exist_ok=True)
DATA_ROOT = STORAGE / "dataset"
RUNS_ROOT = STORAGE / "runs"
CONFIG_PATH = STORAGE / "config.yaml"

if USE_WANDB:
    import wandb

    wandb.login()
print(f"storage: {STORAGE}")

## Download LOCO

The script skips the download when the dataset is already in place, so re-running a later session
costs nothing.

In [ ]:
!bash scripts/download_loco.sh {DATA_ROOT}

## Write the configuration

`data.raw_dir` names the directory the download script wrote; it holds `rgb/` and `subset-*/`.

In [ ]:
import yaml

config = yaml.safe_load(Path("config.yaml").read_text())
config["data"]["raw_dir"] = str(DATA_ROOT)
config["output"] = str(RUNS_ROOT / "baseline")
config["wandb"]["enabled"] = USE_WANDB
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG_PATH.read_text())

## Train and evaluate

Both cells call the repository entrypoints with the platform's own Python.

In [ ]:
!{sys.executable} train.py --config {CONFIG_PATH}

In [ ]:
BEST_WEIGHTS = RUNS_ROOT / "baseline" / "weights" / "best.pt"
!{sys.executable} eval.py --config {CONFIG_PATH} --weights {BEST_WEIGHTS}

## Inspect a prediction

Choose any validation sample and compare its ground-truth boxes with the model predictions.

In [ ]:
SAMPLE_INDEX = 0
SCORE_THRESHOLD = 0.5
!{sys.executable} predict.py --config {CONFIG_PATH} --weights {BEST_WEIGHTS} --index {SAMPLE_INDEX} --num-images 1 --score-threshold {SCORE_THRESHOLD} --show